In [ ]:
# !wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [3]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Point it at the model you just downloaded
base_options = python.BaseOptions(model_asset_path="../model/hand_landmarker.task")

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1774601287.855264   16992 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774601287.967808   17009 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.09), renderer: NVIDIA GeForce RTX 3060 Ti/PCIe/SSE2
W0000 00:00:1774601287.988263   16993 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774601288.003465   17000 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path="model/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # mirror it

    # OpenCV is BGR, MediaPipe wants RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)
    print(result)  # just print for now

    cv2.imshow("cam", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [8]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path="../model/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

# The 21 landmark connections (pairs of indices to draw lines between)
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),         # thumb
    (0,5),(5,6),(6,7),(7,8),         # index
    (0,9),(9,10),(10,11),(11,12),    # middle
    (0,13),(13,14),(14,15),(15,16),  # ring
    (0,17),(17,18),(18,19),(19,20),  # pinky
    (5,9),(9,13),(13,17),            # palm
]

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            # Convert normalised coords to pixel coords
            points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

            # Draw connections
            for a, b in HAND_CONNECTIONS:
                cv2.line(frame, points[a], points[b], (0, 255, 0), 2)

            # Draw landmark dots
            for x, y in points:
                cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

    cv2.imshow("Hand Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1774601377.490196   17137 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774601377.634441   17154 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.09), renderer: NVIDIA GeForce RTX 3060 Ti/PCIe/SSE2
W0000 00:00:1774601377.654798   17139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774601377.669868   17141 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
[ WARN:0@174.726] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ERROR:0@174.726] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
ioctl(VIDIOC_QBUF): Bad file descriptor


In [5]:
cv2.imread("../image/hamster1.webp")

[ WARN:0@143.272] global loadsave.cpp:278 findDecoder imread_('../image/hamster1.webp'): can't open/read file: check file path/integrity


In [20]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

base_options = python.BaseOptions(model_asset_path="../model/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

# Load hamster image
hamster = cv2.imread("../images/hamster1.webp", cv2.IMREAD_UNCHANGED)
hamster = cv2.resize(hamster, (200, 200))

def is_finger_extended(landmarks, tip, mcp):
    return landmarks[tip].y < landmarks[mcp].y

def is_peace_sign(landmarks):
    index_up  = is_finger_extended(landmarks, 8, 5)
    middle_up = is_finger_extended(landmarks, 12, 9)
    ring_down = not is_finger_extended(landmarks, 16, 13)
    pinky_down = not is_finger_extended(landmarks, 20, 17)
    return index_up and middle_up and ring_down and pinky_down

def overlay_image(frame, overlay, x, y):
    h, w = overlay.shape[:2]
    # Clamp to frame bounds
    x1, y1 = max(x, 0), max(y, 0)
    x2, y2 = min(x + w, frame.shape[1]), min(y + h, frame.shape[0])
    ox1 = x1 - x
    oy1 = y1 - y
    ox2 = ox1 + (x2 - x1)
    oy2 = oy1 + (y2 - y1)

    if overlay.shape[2] == 4:  # has alpha channel
        alpha = overlay[oy1:oy2, ox1:ox2, 3:4] / 255.0
        frame[y1:y2, x1:x2] = (
            overlay[oy1:oy2, ox1:ox2, :3] * alpha +
            frame[y1:y2, x1:x2] * (1 - alpha)
        ).astype(np.uint8)
    else:
        frame[y1:y2, x1:x2] = overlay[oy1:oy2, ox1:ox2]

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

            for a, b in HAND_CONNECTIONS:
                cv2.line(frame, points[a], points[b], (0, 255, 0), 2)
            for px, py in points:
                cv2.circle(frame, (px, py), 4, (0, 0, 255), -1)

            if is_peace_sign(hand_landmarks):
                # Place hamster above the wrist
                wx, wy = points[0]
                overlay_image(frame, hamster, wx - 100, wy - 250)

    cv2.imshow("Hand Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1774605182.577451   29854 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774605182.654973   29871 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.09), renderer: NVIDIA GeForce RTX 3060 Ti/PCIe/SSE2
W0000 00:00:1774605182.681378   29857 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774605182.687983   29859 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
[ WARN:0@1371.737] global loadsave.cpp:278 findDecoder imread_('../images/hamster1.webp'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [30]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import math

base_options = python.BaseOptions(model_asset_path="../model/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

# Load memes
memes = {
    "peace":    cv2.imread("../images/hamster_peace.webp"),
    "fingergun": cv2.imread("../images/hamster_gun.webp"),
    "fist": cv2.imread("../images/hamster_scream.jpg"),
    "thumbsup": cv2.imread("../images/hamster_thumbs_up.jpg"),  # swap for your thumbs up image
}

# Resize all memes to same size
for key in memes:
    memes[key] = cv2.resize(memes[key], (400, 400))

# def is_finger_extended(landmarks, tip, mcp):
#     return landmarks[tip].y < landmarks[mcp].y

def angle(a, b, c):
    # angle at point b, between a->b->c
    ax, ay = a.x - b.x, a.y - b.y
    cx, cy = c.x - b.x, c.y - b.y
    dot = ax*cx + ay*cy
    mag = (ax**2 + ay**2)**0.5 * (cx**2 + cy**2)**0.5
    if mag == 0:
        return 0
    return math.degrees(math.acos(max(-1, min(1, dot/mag))))

def is_finger_extended(landmarks, tip, pip, mcp):
    a = angle(landmarks[mcp], landmarks[pip], landmarks[tip])
    return a > 150  # close to straight

# def detect_gesture(landmarks):
#     index_up  = is_finger_extended(landmarks, 8, 5)
#     middle_up = is_finger_extended(landmarks, 12, 9)
#     ring_up   = is_finger_extended(landmarks, 16, 13)
#     pinky_up  = is_finger_extended(landmarks, 20, 17)
#     thumb_up  = landmarks[4].y < landmarks[3].y < landmarks[2].y  # thumb tip above joints
#     thumb_up  = is_finger_extended(landmarks, 4, 1)
#     if index_up and middle_up and not ring_up and not pinky_up:
#         return "peace"
#     if thumb_up and index_up and not middle_up and not ring_up and not pinky_up:
#         return "fingergun"
#     if not thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
#         return "fist"
#     if thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
#         return "thumbsup"
#     return None
def detect_gesture(landmarks):
    index_up  = is_finger_extended(landmarks, 8, 7, 5)
    middle_up = is_finger_extended(landmarks, 12, 11, 9)
    ring_up   = is_finger_extended(landmarks, 16, 15, 13)
    pinky_up  = is_finger_extended(landmarks, 20, 19, 17)
    thumb_up  = is_finger_extended(landmarks, 4, 3, 2)

    if index_up and middle_up and not ring_up and not pinky_up:
        return "peace"
    if thumb_up and index_up and not middle_up and not ring_up and not pinky_up:
        return "fingergun"
    if not thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
        return "fist"
    if thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
        return "thumbsup"
    return None

cap = cv2.VideoCapture(1)

# Create meme window
cv2.namedWindow("Meme", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Meme", 600, 800)

# Black placeholder for meme window
blank = np.zeros((400, 400, 3), dtype=np.uint8)
cv2.imshow("Meme", blank)

current_gesture = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    gesture = None

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

            for a, b in HAND_CONNECTIONS:
                cv2.line(frame, points[a], points[b], (0, 255, 0), 2)
            for px, py in points:
                cv2.circle(frame, (px, py), 4, (0, 0, 255), -1)

            gesture = detect_gesture(hand_landmarks)

            # Show gesture label on webcam feed
            if gesture:
                cv2.putText(frame, gesture, (10, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    # Only update meme window when gesture changes
    if gesture != current_gesture:
        current_gesture = gesture
        if gesture and gesture in memes:
            cv2.imshow("Meme", memes[gesture])
        else:
            cv2.imshow("Meme", blank)

    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Exception ignored in: <function HandLandmarker.__del__ at 0x737e94d817e0>
Traceback (most recent call last):
  File "/home/fatduck/.local/lib/python3.10/site-packages/mediapipe/tasks/python/vision/hand_landmarker.py", line 603, in __del__
    self.close()
  File "/home/fatduck/.local/lib/python3.10/site-packages/mediapipe/tasks/python/vision/hand_landmarker.py", line 579, in close
    self._lib.MpHandLandmarkerClose(self._handle)
  File "/home/fatduck/.local/lib/python3.10/site-packages/mediapipe/tasks/python/core/serial_dispatcher.py", line 74, in shutdown_aware_handler
    return handler(*args, **kwargs)
  File "/home/fatduck/.local/lib/python3.10/site-packages/mediapipe/tasks/python/core/mediapipe_c_utils.py", line 142, in dispatcher_wrapper
    executor.submit(dispatch_and_free).result()
  File "/usr/lib/python3.10/concurrent/futures/_base.py", line 453, in result
    self._condition.wait(timeout)
  File "/usr/lib/python3.10/threading.py", line 320, in wait
    waiter.acquire()
Key

SyntaxError: invalid decimal literal (1862216070.py, line 87)

In [31]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import math

base_options = python.BaseOptions(model_asset_path="../model/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]

# Load memes
memes = {
    "peace":    cv2.imread("../images/hamster_peace.webp"),
    "fingergun": cv2.imread("../images/hamster_gun.webp"),
    "fist": cv2.imread("../images/hamster_scream.jpg"),
    "flat": cv2.imread("../images/hamster_flat.webp"),
    "thumbsup": cv2.imread("../images/hamster_thumbs_up.jpg"),  # swap for your thumbs up image
}

# Resize all memes to same size
# for key in memes:
#     memes[key] = cv2.resize(memes[key], (400, 800))

def angle(a, b, c):
    # angle at point b, between a->b->c
    ax, ay = a.x - b.x, a.y - b.y
    cx, cy = c.x - b.x, c.y - b.y
    dot = ax*cx + ay*cy
    mag = (ax**2 + ay**2)**0.5 * (cx**2 + cy**2)**0.5
    if mag == 0:
        return 0
    return math.degrees(math.acos(max(-1, min(1, dot/mag))))

def is_finger_extended(landmarks, tip, pip, mcp):
    a = angle(landmarks[mcp], landmarks[pip], landmarks[tip])
    return a > 150  # close to straight

def detect_gesture(landmarks):
    index_up  = is_finger_extended(landmarks, 8, 7, 5)
    middle_up = is_finger_extended(landmarks, 12, 11, 9)
    ring_up   = is_finger_extended(landmarks, 16, 15, 13)
    pinky_up  = is_finger_extended(landmarks, 20, 19, 17)
    thumb_up  = is_finger_extended(landmarks, 4, 3, 2)

    if index_up and middle_up and not ring_up and not pinky_up:
        return "peace"
    elif thumb_up and index_up and not middle_up and not ring_up and not pinky_up:
        return "fingergun"
    elif not thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
        return "fist"
    elif thumb_up and not index_up and not middle_up and not ring_up and not pinky_up:
        return "thumbsup"
    elif thumb_up and index_up and middle_up and ring_up and pinky_up:
        return "flat"
    
    return None

cap = cv2.VideoCapture(1)

# Create meme window
cv2.namedWindow("Meme", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Meme", 600, 800)

# Black placeholder for meme window
blank = np.zeros((400, 400, 3), dtype=np.uint8)
cv2.imshow("Meme", blank)

current_gesture = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    gesture = None

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

            for a, b in HAND_CONNECTIONS:
                cv2.line(frame, points[a], points[b], (0, 255, 0), 2)
            for px, py in points:
                cv2.circle(frame, (px, py), 4, (0, 0, 255), -1)

            gesture = detect_gesture(hand_landmarks)

            # Show gesture label on webcam feed
            if gesture:
                cv2.putText(frame, gesture, (10, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    # Only update meme window when gesture changes
    if gesture != current_gesture:
        current_gesture = gesture
        if gesture and gesture in memes:
            cv2.imshow("Meme", memes[gesture])
        else:
            cv2.imshow("Meme", blank)

    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1774605638.760584   30806 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774605638.861653   30823 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.09), renderer: NVIDIA GeForce RTX 3060 Ti/PCIe/SSE2
W0000 00:00:1774605638.882511   30809 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774605638.893581   30810 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
[ WARN:0@1827.968] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
ioctl(VIDIOC_QBUF): Bad file descriptor
